# Oléfines – Décomposition FCC proxy / Steam Cracker (module *fcc-cracker*)

Ce notebook lit deux jeux de scénarios :
- **Total base** (périmètre global oléfines, potentiellement steam cracker + raffinerie)  
- **Proxy raffinerie / FCC (CONCAWE)**

Puis il reconstruit un résiduel **steam cracker** par différence, en assurant une identité additive:

\[ \text{steamcracker}(t) = \max(0,\ \text{total}(t) - \text{refinery\_proxy}(t)) \]

et, pour garantir l'identité en sortie :

\[ \text{refinery\_adj}(t)=\min(\text{refinery\_proxy}(t),\ \text{total}(t)) \]  
\[ \text{steamcracker}(t)=\text{total}(t)-\text{refinery\_adj}(t) \]

> **Important unités** : les deux entrées doivent être comparables (idéalement *t/an*).  
Si `olefins_production` n'est pas en tonnes, vous devez soit :
- convertir via un facteur explicite (paramètre `units.scale_total_to_tonnes`), soit
- accepter que le split soit « indiciel » et non massique.



In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# =============================================================================
# CONFIG
# =============================================================================
# 👉 Mettez ici le chemin de votre YAML "fcc-cracker"
CONFIG_PATH = Path("olefins_fcc-cracker_config.yaml")

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

YEAR_START = int(CFG["years"]["start"])
YEAR_END = int(CFG["years"]["end"])
YEARS = np.arange(YEAR_START, YEAR_END + 1)

INP = CFG["inputs"]
MAP = CFG["scenario_mapping"]["total_to_refinery"]
OUT = CFG["output"]
CHECKS = CFG.get("checks", {})

OUT_DIR = Path(OUT["base_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional unit scaling (defaults to 1.0)
UNITS = CFG.get("units", {})
SCALE_TOTAL_TO_TONNES = float(UNITS.get("scale_total_to_tonnes", 1.0))
SCALE_PROXY_TO_TONNES = float(UNITS.get("scale_proxy_to_tonnes", 1.0))

print("Loaded config:", CONFIG_PATH.resolve())
print(f"Years: {YEAR_START}-{YEAR_END} (n={len(YEARS)})")
print(f"Scaling: total×{SCALE_TOTAL_TO_TONNES}, proxy×{SCALE_PROXY_TO_TONNES}")


In [ ]:
# =============================================================================
# HELPERS
# =============================================================================

def _assert_cols(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}. Available: {list(df.columns)}")

def load_total_base_scenario(base_dir: Path, scenario_dir: str, file_name: str, activity_col: str) -> pd.DataFrame:
    """Load total-base file (one scenario). Expected columns at least: country, year, <activity_col>."""
    path = base_dir / scenario_dir / file_name
    df = pd.read_csv(path)
    _assert_cols(df, {"country", "year", activity_col}, f"total_base:{scenario_dir} ({path})")

    out = df[["country", "year", activity_col]].copy()
    out["country"] = out["country"].astype(str).str.upper()
    out["year"] = pd.to_numeric(out["year"], errors="coerce").astype(int)
    out[activity_col] = pd.to_numeric(out[activity_col], errors="coerce").fillna(0.0)

    out = out[out["year"].isin(YEARS)].copy()
    out = out.rename(columns={activity_col: "olefins_total_base_raw"})
    return out

def load_refinery_proxy(base_dir: Path, refinery_scenario: str, demand_scenario: str, file_name: str, activity_col: str) -> pd.DataFrame:
    """Load CONCAWE refinery-proxy file for a given refinery_scenario/demand_scenario."""
    path = base_dir / refinery_scenario / demand_scenario / file_name
    df = pd.read_csv(path)
    _assert_cols(df, {"country", "year", activity_col}, f"refinery_proxy:{refinery_scenario}/{demand_scenario} ({path})")

    out = df[["country", "year", activity_col]].copy()
    out["country"] = out["country"].astype(str).str.upper()
    out["year"] = pd.to_numeric(out["year"], errors="coerce").astype(int)
    out[activity_col] = pd.to_numeric(out[activity_col], errors="coerce").fillna(0.0)

    out = out[out["year"].isin(YEARS)].copy()
    out = out.rename(columns={activity_col: "olefins_refinery_proxy_raw"})
    return out

def compute_split(df: pd.DataFrame) -> pd.DataFrame:
    """Apply scaling + strict additive decomposition."""
    df = df.copy()

    # Scale (if needed)
    df["olefins_total_base_t_per_yr"] = df["olefins_total_base_raw"] * SCALE_TOTAL_TO_TONNES
    df["olefins_refinery_proxy_t_per_yr"] = df["olefins_refinery_proxy_raw"] * SCALE_PROXY_TO_TONNES

    # Strict identity: refinery_adj = min(proxy, total)
    df["olefins_refinery_t_per_yr"] = np.minimum(
        df["olefins_refinery_proxy_t_per_yr"],
        df["olefins_total_base_t_per_yr"],
    )
    df["olefins_steamcracker_t_per_yr"] = (
        df["olefins_total_base_t_per_yr"] - df["olefins_refinery_t_per_yr"]
    )

    # Shares (avoid division by 0)
    denom = df["olefins_total_base_t_per_yr"].replace(0.0, np.nan)
    df["share_refinery"] = (df["olefins_refinery_t_per_yr"] / denom).clip(lower=0.0, upper=1.0).fillna(0.0)
    df["share_steamcracker"] = (1.0 - df["share_refinery"]).clip(lower=0.0, upper=1.0)

    # Diagnostics
    df["proxy_exceeds_total"] = df["olefins_refinery_proxy_t_per_yr"] > df["olefins_total_base_t_per_yr"] + 1e-9

    return df

def run_checks(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("Output dataframe is empty (check inputs and years filter).")

    if CHECKS.get("non_negative", True):
        cols = [
            "olefins_total_base_t_per_yr",
            "olefins_refinery_proxy_t_per_yr",
            "olefins_refinery_t_per_yr",
            "olefins_steamcracker_t_per_yr",
            "share_refinery",
            "share_steamcracker",
        ]
        bad = (df[cols] < -1e-9).any().any()
        if bad:
            raise ValueError("Negative values detected in output (unexpected).")

    if CHECKS.get("full_year_coverage", True):
        expected = len(YEARS)
        for c, g in df.groupby("country"):
            if len(g["year"].unique()) != expected:
                missing = sorted(set(YEARS) - set(g["year"].unique()))
                raise ValueError(f"Incomplete year coverage for {c}. Missing years: {missing[:10]}{'...' if len(missing)>10 else ''}")

    if CHECKS.get("enforce_mass_balance_identity", True):
        diff = df["olefins_total_base_t_per_yr"] - (
            df["olefins_refinery_t_per_yr"] + df["olefins_steamcracker_t_per_yr"]
        )
        if (diff.abs() > 1e-6).any():
            raise ValueError("Mass balance identity violation detected.")

    if CHECKS.get("warn_if_proxy_exceeds_total", True):
        n = int(df["proxy_exceeds_total"].sum())
        if n > 0:
            # Not an error: just a warning (identity holds because we cap refinery at total).
            print(f"WARNING: proxy_exceeds_total for {n} rows (refinery proxy > total base). "
                  f"Steamcracker set to 0 and refinery capped to total for those rows.")


In [ ]:
# =============================================================================
# MAIN
# =============================================================================

total_base_dir = Path(INP["total_base"]["base_dir"])
total_scen_dirs = INP["total_base"]["scenario_dirs"]          # dict like {bau-eu: "bau-eu", ...}
total_file = INP["total_base"]["file_name"]
total_col = INP["total_base"]["activity_column"]

proxy_base_dir = Path(INP["refinery_proxy"]["base_dir"])
proxy_file = INP["refinery_proxy"]["file_name"]
proxy_col = INP["refinery_proxy"]["activity_column"]

rows_written = 0

for total_scenario, total_rel in total_scen_dirs.items():
    # Map to refinery scenario (more-molecule / max-electron)
    refinery_scenario = MAP[total_scenario]

    # We will iterate over ALL demand_scenarios present under refinery_proxy/<refinery_scenario>/...
    # (since olefins-concawe writes: refinery_scenario/demand_scenario/olefins.csv)
    ref_scen_dir = proxy_base_dir / refinery_scenario
    if not ref_scen_dir.exists():
        raise FileNotFoundError(f"Missing refinery proxy directory: {ref_scen_dir}")

    demand_scenarios = sorted([p.name for p in ref_scen_dir.iterdir() if p.is_dir()])
    if not demand_scenarios:
        raise ValueError(f"No demand_scenario folders found under: {ref_scen_dir}")

    # Load total base once per total scenario
    df_total = load_total_base_scenario(total_base_dir, total_rel, total_file, total_col)

    for demand_scenario in demand_scenarios:
        df_proxy = load_refinery_proxy(proxy_base_dir, refinery_scenario, demand_scenario, proxy_file, proxy_col)

        # Join on (country,year)
        df = df_total.merge(df_proxy, on=["country", "year"], how="outer").fillna(0.0)

        df["total_scenario"] = total_scenario
        df["refinery_scenario"] = refinery_scenario
        df["demand_scenario_refinery_proxy"] = demand_scenario

        df = compute_split(df)

        # Checks
        run_checks(df)

        # Output selection/rename according to YAML output.columns
        cols_out = OUT["columns"]

        out_df = pd.DataFrame({
            cols_out["total_scenario"]: df["total_scenario"],
            cols_out["refinery_scenario"]: df["refinery_scenario"],

            cols_out["olefins_total_base"]: df["olefins_total_base_t_per_yr"],
            cols_out["olefins_refinery_proxy"]: df["olefins_refinery_proxy_t_per_yr"],
            cols_out["olefins_steamcracker"]: df["olefins_steamcracker_t_per_yr"],

            cols_out["share_refinery"]: df["share_refinery"],
            cols_out["share_steamcracker"]: df["share_steamcracker"],
        })

        # Write
        out_path = OUT_DIR / total_scenario / refinery_scenario
        out_path.mkdir(parents=True, exist_ok=True)

        out_file = out_path / OUT["output_file_name"]
        out_df.to_csv(out_file, index=False)

        rows_written += len(out_df)

        print(f"Wrote {len(out_df):,} rows → {out_file} (proxy demand_scenario={demand_scenario})")

print(f"\nDONE. Total rows written: {rows_written:,}")
